In [1]:
import pandas as pd
import epi_utils as eu

In [2]:
from sqlalchemy import create_engine
import pymysql

In [3]:
from pandasql import sqldf

# Define a reusable function for running SQL queries
run_query = lambda query: sqldf(query, globals())

In [4]:
from tqdm.auto import tqdm
tqdm.pandas()

In [5]:
username = "root"
password = ""
port = 3306
database = "hg19"

In [6]:
engine = create_engine('mysql+pymysql://%s@localhost:%i/%s' %(username, port, database))

In [7]:
sql = "SELECT * FROM ncbirefseq"
ncbi_df = pd.read_sql_query(sql, engine)

display(ncbi_df.head())
display(ncbi_df.shape)

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'","b'12227,12721,14409,'",0,DDX11L1,none,none,"b'-1,-1,-1,'"
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...","b'14829,15038,15947,16765,17055,17368,17742,18...",0,WASH7P,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'"
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'","b'17436,'",0,MIR6859-1,none,none,"b'-1,'"
3,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'","b'30503,'",0,MIR1302-2,none,none,"b'-1,'"
4,585,NR_026818.1,chr1,-,34610,36081,36081,36081,3,"b'34610,35276,35720,'","b'35174,35481,36081,'",0,FAM138A,none,none,"b'-1,-1,-1,'"


(91315, 16)

In [8]:
ncbi_df.loc[:, "tss"] = ncbi_df.progress_apply(lambda row: eu.get_tss_ncbi(row), axis=1)

  0%|          | 0/91315 [00:00<?, ?it/s]

In [9]:
display(ncbi_df.head())

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames,tss
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'","b'12227,12721,14409,'",0,DDX11L1,none,none,"b'-1,-1,-1,'",11873
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...","b'14829,15038,15947,16765,17055,17368,17742,18...",0,WASH7P,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'",29370
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'","b'17436,'",0,MIR6859-1,none,none,"b'-1,'",17436
3,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'","b'30503,'",0,MIR1302-2,none,none,"b'-1,'",30365
4,585,NR_026818.1,chr1,-,34610,36081,36081,36081,3,"b'34610,35276,35720,'","b'35174,35481,36081,'",0,FAM138A,none,none,"b'-1,-1,-1,'",36081


In [10]:
histone_df = eu.load_histone_files()
display(histone_df.head())
display(histone_df.shape)

,chrom,chromStart,chromEnd,name,length,type
0,chr10,119808,119954,chr10_173,146,h3k4me3
1,chr10,119956,120102,chr10_174,146,h3k4me3
2,chr10,122100,122246,chr10_185,146,h3k4me3
3,chr10,122308,122454,chr10_186,146,h3k4me3
4,chr10,180346,180492,chr10_489,146,h3k4me3


(319086, 6)

In [16]:
ncbi_df.loc[:, "histone_count_dict"] = ncbi_df.progress_apply(lambda row: eu.count_histone(histone_df, row), axis = 1)

  0%|          | 0/91315 [00:00<?, ?it/s]

In [17]:
display(ncbi_df.head())

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames,tss,histone_count_dict
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'","b'12227,12721,14409,'",0,DDX11L1,none,none,"b'-1,-1,-1,'",11873,"{'h3k9me3': 3, 'h3k4me3': 1, 'h3k27ac': 1, 'h3..."
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...","b'14829,15038,15947,16765,17055,17368,17742,18...",0,WASH7P,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'",29370,"{'h3k27me3': 3, 'h3k4me3': 2, 'h3k9ac': 2, 'h3..."
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'","b'17436,'",0,MIR6859-1,none,none,"b'-1,'",17436,{'histone_count_total': 0}
3,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'","b'30503,'",0,MIR1302-2,none,none,"b'-1,'",30365,"{'h3k27me3': 3, 'h3k4me3': 2, 'h3k9ac': 2, 'h3..."
4,585,NR_026818.1,chr1,-,34610,36081,36081,36081,3,"b'34610,35276,35720,'","b'35174,35481,36081,'",0,FAM138A,none,none,"b'-1,-1,-1,'",36081,"{'h3k4me3': 1, 'h3k9ac': 1, 'h3k27me3': 1, 'hi..."


In [18]:
# Normalize the JSON column
ncbi_df = pd.concat([ncbi_df, pd.json_normalize(ncbi_df['histone_count_dict'])], axis = 1)
display(ncbi_df.head())

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,...,cdsEndStat,exonFrames,tss,histone_count_dict,h3k9me3,h3k4me3,h3k27ac,h3k27me3,histone_count_total,h3k9ac
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'",...,none,"b'-1,-1,-1,'",11873,"{'h3k9me3': 3, 'h3k4me3': 1, 'h3k27ac': 1, 'h3...",3.0,1.0,1.0,1.0,6,NaN
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...",...,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'",29370,"{'h3k27me3': 3, 'h3k4me3': 2, 'h3k9ac': 2, 'h3...",1.0,2.0,1.0,3.0,9,2.0
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'",...,none,"b'-1,'",17436,{'histone_count_total': 0},NaN,NaN,NaN,NaN,0,NaN
3,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'",...,none,"b'-1,'",30365,"{'h3k27me3': 3, 'h3k4me3': 2, 'h3k9ac': 2, 'h3...",1.0,2.0,1.0,3.0,9,2.0
4,585,NR_026818.1,chr1,-,34610,36081,36081,36081,3,"b'34610,35276,35720,'",...,none,"b'-1,-1,-1,'",36081,"{'h3k4me3': 1, 'h3k9ac': 1, 'h3k27me3': 1, 'hi...",NaN,1.0,NaN,1.0,3,1.0


In [19]:
# Drop the dict column as it no longer needed
ncbi_df.drop(columns=["histone_count_dict"], inplace=True)
ncbi_df.fillna(0, inplace=True)
display(ncbi_df.head())

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,...,cdsStartStat,cdsEndStat,exonFrames,tss,h3k9me3,h3k4me3,h3k27ac,h3k27me3,histone_count_total,h3k9ac
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'",...,none,none,"b'-1,-1,-1,'",11873,3.0,1.0,1.0,1.0,6,0.0
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...",...,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'",29370,1.0,2.0,1.0,3.0,9,2.0
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'",...,none,none,"b'-1,'",17436,0.0,0.0,0.0,0.0,0,0.0
3,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'",...,none,none,"b'-1,'",30365,1.0,2.0,1.0,3.0,9,2.0
4,585,NR_026818.1,chr1,-,34610,36081,36081,36081,3,"b'34610,35276,35720,'",...,none,none,"b'-1,-1,-1,'",36081,0.0,1.0,0.0,1.0,3,1.0


In [20]:
# Correcting the data type
ncbi_df = ncbi_df.astype({'h3k4me3': 'int32', 'h3k9ac':'int32', 'h3k9me3':'int32', 'h3k27ac':'int32', 'h3k27me3':'int32', 'histone_count_total':'int32'})
display(ncbi_df.head())

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,...,cdsStartStat,cdsEndStat,exonFrames,tss,h3k9me3,h3k4me3,h3k27ac,h3k27me3,histone_count_total,h3k9ac
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'",...,none,none,"b'-1,-1,-1,'",11873,3,1,1,1,6,0
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...",...,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'",29370,1,2,1,3,9,2
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'",...,none,none,"b'-1,'",17436,0,0,0,0,0,0
3,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'",...,none,none,"b'-1,'",30365,1,2,1,3,9,2
4,585,NR_026818.1,chr1,-,34610,36081,36081,36081,3,"b'34610,35276,35720,'",...,none,none,"b'-1,-1,-1,'",36081,0,1,0,1,3,1


In [21]:
display(ncbi_df.columns)

Index(['bin', 'name', 'chrom', 'strand', 'txStart', 'txEnd', 'cdsStart',
       'cdsEnd', 'exonCount', 'exonStarts', 'exonEnds', 'score', 'name2',
       'cdsStartStat', 'cdsEndStat', 'exonFrames', 'tss', 'h3k9me3', 'h3k4me3',
       'h3k27ac', 'h3k27me3', 'histone_count_total', 'h3k9ac'],
      dtype='object')

In [22]:
ncbi_df = ncbi_df[['bin', 'name', 'chrom', 'strand', 'txStart', 'txEnd', 'cdsStart',
       'cdsEnd', 'exonCount', 'exonStarts', 'exonEnds', 'score', 'name2',
       'cdsStartStat', 'cdsEndStat', 'exonFrames', 'tss', 'h3k4me3',
       'h3k9ac', 'h3k27ac', 'h3k27me3', 'h3k9me3', 'histone_count_total']]

display(ncbi_df.head())

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,...,cdsStartStat,cdsEndStat,exonFrames,tss,h3k4me3,h3k9ac,h3k27ac,h3k27me3,h3k9me3,histone_count_total
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'",...,none,none,"b'-1,-1,-1,'",11873,1,0,1,1,3,6
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...",...,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'",29370,2,2,1,3,1,9
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'",...,none,none,"b'-1,'",17436,0,0,0,0,0,0
3,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'",...,none,none,"b'-1,'",30365,2,2,1,3,1,9
4,585,NR_026818.1,chr1,-,34610,36081,36081,36081,3,"b'34610,35276,35720,'",...,none,none,"b'-1,-1,-1,'",36081,1,1,0,1,0,3


In [29]:
ncbi_df.describe()

,bin,txStart,txEnd,cdsStart,cdsEnd,exonCount,score,tss,h3k4me3,h3k9ac,h3k27ac,h3k27me3,h3k9me3,histone_count_total
count,91315.000000,9.131500e+04,9.131500e+04,9.131500e+04,9.131500e+04,91315.000000,91315.0,9.131500e+04,91315.000000,91315.000000,91315.000000,91315.000000,91315.000000,91315.000000
mean,742.127449,6.868001e+07,6.874931e+07,6.869640e+07,6.874194e+07,10.645228,0.0,6.871428e+07,5.439774,4.552330,4.460078,0.664009,0.731369,15.847561
std,575.466184,5.753683e+07,5.754871e+07,5.753942e+07,5.754749e+07,9.361598,0.0,5.754206e+07,4.870041,4.179203,4.272305,1.019728,0.990109,12.661730
min,0.000000,0.000000e+00,7.690000e+02,0.000000e+00,4.180000e+02,1.000000,0.0,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,163.000000,2.196744e+07,2.200526e+07,2.199062e+07,2.199986e+07,4.000000,0.0,2.199585e+07,1.000000,1.000000,1.000000,0.000000,0.000000,5.000000
50%,687.000000,5.361113e+07,5.363616e+07,5.361163e+07,5.361982e+07,8.000000,0.0,5.363568e+07,5.000000,4.000000,4.000000,0.000000,0.000000,14.000000
75%,1098.000000,1.058567e+08,1.058817e+08,1.058568e+08,1.058799e+08,14.000000,0.0,1.058567e+08,8.000000,7.000000,7.000000,1.000000,1.000000,23.000000
max,2486.000000,2.492004e+08,2.492133e+08,2.492115e+08,2.492126e+08,363.000000,0.0,2.492004e+08,34.000000,30.000000,35.000000,10.000000,11.000000,99.000000


In [28]:
ncbi_df.to_csv("dataset/ncbi_histone_count.csv", header=True, index=False)

# HepG2 Data

In [30]:
hepg2_df = pd.read_csv("dataset/GSM3718064_HepG2_exp.txt", sep="\t")
display(hepg2_df.head())

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no


In [31]:
# Splitting chromosome name and the position
hepg2_df[["chrom", "chromPos"]] = hepg2_df["locus"].str.split(':', expand=True)
display(hepg2_df.head())

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chrom,chromPos
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,69090-70008
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no,chr1,323891-328581
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,367658-368597
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no,chr1,761585-794889
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,840263-843900


In [32]:
# Splitting start and end position
hepg2_df[["chromStart", "chromEnd"]] = hepg2_df["chromPos"].str.split('-', expand=True)
display(hepg2_df.head())

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chrom,chromPos,chromStart,chromEnd
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,69090-70008,69090,70008
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no,chr1,323891-328581,323891,328581
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,367658-368597,367658,368597
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no,chr1,761585-794889,761585,794889
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,840263-843900,840263,843900


In [33]:
# Correcting the Data Types
hepg2_df = hepg2_df.astype({'chromStart': 'int32', 'chromEnd': 'int32'})

In [34]:
hepg2_df = hepg2_df[['test_id', 'gene_id', 'gene', 'locus', 'sample_1', 'sample_2',
       'status', 'value_1', 'value_2', 'log2(fold_change)', 'test_stat',
       'p_value', 'q_value', 'significant', 'chrom', 'chromStart', 'chromEnd']]

display(hepg2_df.head())
display(hepg2_df.shape)

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chrom,chromStart,chromEnd
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,69090,70008
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no,chr1,323891,328581
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,367658,368597
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no,chr1,761585,794889
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,840263,843900


(30032, 17)

In [ ]:
hepg2_df.to_csv("dataset/hepg2_exp_transformed.csv", header=True, index=False)

In [48]:
# Join the dataset

q_join_hepg2_ncbi = '''
    SELECT h.*, n.*
    FROM hepg2_df h
    JOIN ncbi_df n
    ON h.chrom = n.chrom
    AND h.chromStart = n.txStart
    AND h.chromEnd = n.txEnd
'''

result_1 = run_query(q_join_hepg2_ncbi)
display(result_1.head())
display(result_1.shape)

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),...,cdsStartStat,cdsEndStat,exonFrames,tss,h3k4me3,h3k9ac,h3k27ac,h3k27me3,h3k9me3,histone_count_total
0,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,...,none,none,"b'-1,-1,-1,'",323891,9,1,0,6,1,17
1,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,...,cmpl,cmpl,"b'0,'",367658,1,0,0,0,1,2
2,XLOC_000053,XLOC_000053,LOC284661,chr1:4472110-4484744,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,...,none,none,"b'-1,-1,-1,-1,-1,-1,'",4472110,9,9,9,1,2,30
3,XLOC_000109,XLOC_000109,PRAMEF1,chr1:12851545-12856777,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,...,cmpl,cmpl,"b'-1,0,2,2,'",12851545,4,2,4,0,1,11
4,XLOC_000110,XLOC_000110,PRAMEF2,chr1:12916940-12921764,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,...,cmpl,cmpl,"b'-1,0,2,2,'",12916940,7,2,2,1,0,12


(2188, 40)

In [47]:
result_1.to_csv("dataset/hepg2_ncbi_histone_count.csv", index=False, header=True)